# Electromagnetic Solver Comparison — Darwin Model

The **Darwin model** approximates the full Maxwell equations by neglecting the transverse displacement current in Ampère's law, retaining EM effects at sub-light speeds. Two solver variants are compared:

- **DKPol** — includes the polarisation drift
- **DKNoPol** — omits the polarisation drift

Both solvers are tested at electron-to-ion mass ratios $\mu = m_e/m_i \in \{0.1,\,0.001\}$. The wave dispersion $\omega(k)$ extracted from the simulated $E_y$ field is compared against the cold-plasma analytic result. Sections 1–3 cover solver comparison, time-step convergence, and geometry sensitivity.

In [ ]:
ENV["OMP_NUM_THREADS"] = "4"
using Pkg
Pkg.activate(".")
Pkg.develop(path="..")

isCuda = try
    success(`nvidia-smi`)
catch
    false
end
if isCuda
    println("CUDA is available, using GPU acceleration.")
    using CUDA
end

using bslLD, Random, FFTW, DSP, CairoMakie, Statistics
bslLD.greet()

if isCuda
    println("Setting backend to CUDA.")
    bslLD.use_cuda!()
else
    println("CUDA not available, using CPU.")
end;

In [ ]:
# --- Diagnostics ---
mutable struct Diag
    Ey::Vector   # per-step snapshots of the Ey field
    Bz::Vector   # per-step snapshots of the Ey field
    t::Vector    # corresponding simulation times
end
Diag() = Diag([], [],[])

function record!(diag::Diag, sol, simTime)
    push!(diag.Ey, copy(sol.E[2].data))
    push!(diag.Bz, copy(sol.B[3].data))
    push!(diag.t,  simTime.current_T)
end

# --- Strang split stepper: works for any <: AbstractFieldSolver ---
# V(dt/2, E^n) - X(dt/2) - midpoint moments - solve_fields!(dt) - X(dt/2) - V(dt/2, E^{n+1})
function step!(f_i, sol, grid, simTime, solver)
    phase_start = simTime.phase
    Ω  = simTime.gyro_frequency
    dt = simTime.dt
    Pi_zero = bslLD.zero_vectorfield3(grid)

    # V(dt/2) with E^n
    simTime.fraction_dt = 0.5
    simTime.phase = phase_start + 0.25 * Ω * dt
    bslLD.advectV!(f_i, grid, simTime, sol.E)

    # X(dt/2)
    simTime.phase = phase_start + 0.25 * Ω * dt
    simTime.fraction_dt = 0.5
    bslLD.advectX!(f_i, grid, simTime)

    simTime.phase = phase_start + 0.5 * Ω * dt
    n_i    = bslLD.compute_density(f_i, grid)
    J_perp = bslLD.compute_current(f_i, grid, simTime.phase)
    moments_mid = bslLD.Moments(n_i, J_perp, Pi_zero)

    bslLD.solve_fields!(sol, moments_mid, grid, solver, dt)

    for (ind, Ec) in enumerate(sol.Enew)
        sol.Enew[ind].data .= bslLD.fourier_filter(Ec, grid, 0.7).data
    end

    # X(dt/2)
    simTime.phase = phase_start + 0.75 * Ω * dt
    simTime.fraction_dt = 0.5
    bslLD.advectX!(f_i, grid, simTime)

    # V(dt/2) with E^{n+1}
    simTime.phase = phase_start + 0.75 * Ω * dt
    simTime.fraction_dt = 0.5
    bslLD.advectV!(f_i, grid, simTime, sol.Enew)

    sol.E .= sol.Enew
    simTime.phase = phase_start
    simTime.fraction_dt = 1.0
end

function make_ics(grid, epsilon, T, Nx; random_E = false)
    if random_E
        E0 = bslLD.VectorField([bslLD.ScalarField(epsilon .* randn(Nx)) for _ in 1:3])
    else

        Ey = epsilon .* randn(Nx)
        Ey .-= mean(Ey)
        E0 = bslLD.VectorField([
            bslLD.ScalarField(zeros(Nx)),
            bslLD.ScalarField(Ey),
            bslLD.ScalarField(zeros(Nx)),
        ])
    end



    B0  = bslLD.VectorField([bslLD.ScalarField(zeros(Nx)) for _ in 1:3])
    sol = bslLD.FieldSolution(E0, B0)
    initFuncv(v) = exp(-(v +epsilon*rand())^2 / (2T)) / sqrt(2pi*T)
    f_i = bslLD.Distribution(grid, 0.0000,
                              initFuncx = x -> 1.0,
                              initFuncv = initFuncv)
    return f_i, sol
end

# --- Spectral diagnostic ---
# Returns (k, omega, log-power-spectrum, nk, nw)
function make_spec(diag, dt, grid, Nx)
    omega = fftfreq(length(diag.t), 1 / dt) .* 2pi
    k     = fftfreq(Nx, 1 / grid.delta[1]) .* 2pi
    nw    = length(diag.t) ÷ 2
    nk    = Nx ÷ 2
    w_win = kaiser(length(diag.t), 3)
    data  = transpose(hcat(diag.Bz...))
    spec  = log.(abs.(fft(data .* w_win))[1:nw, 1:nk] .+ 1e-30)
    return k, omega, spec, nk, nw
end

# --- Simulation runner ---
function run_simulation(solver, mu, dt, grid; random_E = false)
    Random.seed!(42)
    f_i, sol  = make_ics(grid, epsilon, T, Nx; random_E)
    solver_fi = solver(beta_i, mu)
    simTime   = bslLD.SimulationTime(dt, Tmax)
    bslLD.ProgressMeter.ijulia_behavior(:clear)
    diag  = Diag()
    while bslLD.continue_advection(simTime, true)
        step!(f_i, sol, grid, simTime, solver_fi)
        bslLD.advance!(simTime)
        record!(diag, sol, simTime)
    end
    diag
end;

In [ ]:
include("cold_plasma.jl")

ws = range(1e-4, 20, length=4000)
kana(w,j,F) = begin
        z = F(w)[j]
        abs(imag(z)) < 1e-8 && real(z) > 0 ? sqrt(real(z)) : NaN
    end;

In [ ]:
function run_simulation_cold(solver, mu, dt, grid; random_E = false)
    Random.seed!(42)
    f_i, sol  = make_ics(grid, 0.0000001, T, Nx; random_E)
    fluid = bslLD.ColdIonFluid(grid)
    solver_fi = solver(beta_i, mu)
    simTime   = bslLD.SimulationTime(dt, Tmax)
    bslLD.ProgressMeter.ijulia_behavior(:clear)
    diag  = Diag()

    while bslLD.continue_advection(simTime, true)
        bslLD.step_cold!(fluid, sol, grid, simTime, solver_fi)
        bslLD.advance!(simTime)
        record!(diag, sol, simTime)
    end
    diag
end;

### Cold-Plasma Baseline ($\mu=0.1$)

In the **cold-plasma limit** the ion temperature is set to $T_i\to 0$ (vanishingly small perturbation $\epsilon=10^{-7}$) so kinetic effects are negligible. The resulting $\omega$–$k$ spectrum should match the analytic cold-plasma dispersion relation (dashed curves) derived from the Darwin dielectric tensor. Parameters: $\beta_i=0.05$, $\mu=0.1$, $\Delta t=0.005$, $T_{\rm end}=76$, propagation direction $k_z$.

In [ ]:
beta_i  = 0.05
mu      = 0.1
epsilon = 1e-7
T       = .1

Lx   = 50pi
Nx   = 128
Nv   = 32
vmax = 6 * sqrt(T)

dt = 0.005
Tmax = 76

grid    = bslLD.Grid([0.0, -vmax, -vmax], [Lx, vmax, vmax], [Nx, Nv, Nv], 1, 1.0, 3)

resultsCold1 = [(mu, string(solver), run_simulation_cold(solver, mu, dt, grid))
            for solver in [bslLD.EMSolverDKPol,bslLD.EMSolverDKNoPol]];

In [ ]:
fig = Figure(size = (900, 400))

for (ind, (mu, label, result)) in enumerate(resultsCold1)
    k, omega, spec, nk, nw = make_spec(result, dt, grid, Nx)

    ax = Axis(fig[1,ind], xlabel = "k", ylabel = "ω",limits = ((0, 2), (0, 20)))
    heatmap!(ax, k[2:nk], omega[2:nw], spec[2:nw, 2:end]')

    model = [K2DriftPol, K2Drift][ind]

    lines!(ax,[kana(w,1,x->model(x,π/2,mu,beta_i)) for w in ws], ws; linestyle = :dash, color = :black)
    lines!(ax,[kana(w,2,x->model(x,π/2,mu,beta_i)) for w in ws], ws; linestyle = :dash, color = :black)
end

fig

### Cold-Plasma Baseline ($\mu=0.1$, $\Delta t$ reduced)

A second cold run with reduced time step ($\Delta t=0.001$, $T_{\rm end}=50$) and $\beta_i=0.1$ probes the propagation direction $k_z$ (index 1). A random $E$-field seed is used to excite all modes simultaneously.

In [ ]:
beta_i  = 0.1
mu      = 0.1
epsilon = 1e-7
T       = 0.1

Lx   = 50pi
Nx   = 128
Nv   = 32
vmax = 6 * sqrt(T)

dt = 0.001
Tmax = 50

grid3 = bslLD.Grid([0.0, -vmax, -vmax], [Lx, vmax, vmax], [Nx, Nv, Nv], 1, 1.0, 1)

resultsCold2 = [run_simulation_cold(solver, mu, dt, grid3; random_E = true)
            for solver in [bslLD.EMSolverDKNoPol, bslLD.EMSolverDKPol]];

In [ ]:

fig = Figure(size = (900, 400))

for (ind, result) in enumerate(resultsCold2)
    k, omega, spec, nk, nw = make_spec(result, dt, grid, Nx)

    ax = Axis(fig[1,ind], xlabel = "k", ylabel = "ω",limits = ((0, 2), (0, 20)))
    heatmap!(ax, k[2:nk], omega[2:nw], spec[2:nw, 2:end]')

    model = [K2Drift,K2DriftPol][ind]

    lines!(ax,[kana(w,1,x->model(x,0,1/10,beta_i)) for w in ws], ws; linestyle = :dash, color = :black)
    lines!(ax,[kana(w,2,x->model(x,0,1/10,beta_i)) for w in ws], ws; linestyle = :dash, color = :black)
end

fig

## 1. Solver comparison

**Kinetic (warm-plasma) runs**: the full Vlasov–Darwin system is solved with gyro-kinetic ions at $T_i=1$, $\beta_i=0.1$, $L_x=25\pi$, propagation direction $\hat{k}\parallel\hat{z}$ (index 3). Both `EMSolverDKPol` (includes polarisation-drift correction) and `EMSolverDKNoPol` (omits it) are run at mass ratios $\mu\in\{0.1,\,0.001\}$.

Box: $L_x = 25\pi$, propagation direction $k_z$ (index 3).

In [ ]:
beta_i  = 0.1
mu      = 0.1
epsilon = 1e-7
T       = 1.0

Lx   = 25pi
Nx   = 128
Nv   = 32
vmax = 6 * sqrt(T)

dt = 0.005
Tmax = 20

grid    = bslLD.Grid([0.0, -vmax, -vmax], [Lx, vmax, vmax], [Nx, Nv, Nv], 1, 1.0, 3)

results1 = [(mu, string(solver), run_simulation(solver, mu, dt, grid))
            for solver in [bslLD.EMSolverDKPol, bslLD.EMSolverDKNoPol]
            for mu    in [0.1, 0.001]];

In [ ]:
fig = Figure(size = (800, 800))
for (ind, (mu, label, result)) in enumerate(results1)
    k, omega, spec, nk, nw = make_spec(result, dt, grid, Nx)
    ax = Axis(fig[cld(ind, 2), mod1(ind, 2)], xlabel = "k", ylabel = "ω",
              title = "$label – μ = $mu")
    nw = nw ÷ 10

    heatmap!(ax, k[2:nk], omega[2:nw], spec[2:2:nw, 2:end]')
end
fig

In [ ]:
fig = Figure(size = (900, 600))
ax  = Axis(fig[1, 1], xlabel = "step", ylabel = "⟨Ey²⟩", yscale = log10)

for (_, label, result) in results1
    Ey2 = map(x -> mean(x .* x), result.Ey)
    plot!(ax, Ey2, label = label)
end
Legend(fig[1, 2], ax)
fig

In [ ]:
beta_i  = 0.05
mu      = 0.1
epsilon = 1e-7
T       = .1

Lx   = 50pi
Nx   = 128
Nv   = 32
vmax = 6 * sqrt(T)

dt = 0.05
Tmax = 50

grid    = bslLD.Grid([0.0, -vmax, -vmax], [Lx, vmax, vmax], [Nx, Nv, Nv], 1, 1.0, 3)

results11 = [(mu, string(solver), run_simulation(solver, mu, dt, grid))
            for solver in [bslLD.EMSolverDKPol,bslLD.EMSolverDKNoPol]];

In [ ]:

fig = Figure(size = (900, 400))

for (ind, (mu, label, result)) in enumerate(results11)
    k, omega, spec, nk, nw = make_spec(result, dt, grid, Nx)

    ax = Axis(fig[1,ind], xlabel = "k", ylabel = "ω",limits = ((0, 2), (0, 15)))
    heatmap!(ax, k[2:nk], omega[2:nw], spec[2:nw, 2:end]')

    model = [K2DriftPol, K2Drift][ind]

    lines!(ax,[kana(w,1,x->model(x,π/2,mu,beta_i)) for w in ws], ws; linestyle = :dash, color = :black)
    lines!(ax,[kana(w,2,x->model(x,π/2,mu,beta_i)) for w in ws], ws; linestyle = :dash, color = :black)
end

fig

In [ ]:
fig = Figure(size = (900, 600))
ax  = Axis(fig[1, 1], xlabel = "step", ylabel = "⟨Ey²⟩", yscale = log10)

for (_, label, result) in results11
    Ey2 = map(x -> mean(x .* x), result.Ey)
    plot!(ax, Ey2, label = label)
end
Legend(fig[1, 2], ax)
fig

## 2. Time convergence study

Vary dt for `EMSolverDKNoPol` (μ = 0.001) and measure convergence against the finest-dt reference.

In [ ]:
mu   = 0.01
dts  = [0.001, 0.0125, 0.025, 0.05, 0.1]
Tmax = 8.0

results2 = [(dt_i, run_simulation(bslLD.EMSolverDKNoPol, mu, dt_i, grid)) for dt_i in dts];

In [ ]:
fig = Figure(size = (700, 400))
ax  = Axis(fig[1, 1], xlabel = "t", ylabel = "⟨Ey²⟩", yscale = log10)

for (dt_i, result) in results2
    simTime = bslLD.SimulationTime(dt_i, Tmax)
    Ey2 = map(x -> mean(x .* x), result.Ey)
    plot!(ax, collect(simTime)[1:length(Ey2)], Ey2, label = "dt = $dt_i")
end
Legend(fig[1, 2], ax)
fig

In [ ]:
fig = Figure()
ax  = Axis(fig[1, 1], xscale = log10, yscale = log10, xlabel = "dt", ylabel = "L1 error in Ey")

ref_Ey = results2[1][2].Ey[end]
error  = [mean(abs.(ref_Ey .- r[2].Ey[end])) for r in results2]

scatter!(ax, dts[2:end], error[2:end])
lines!(ax, dts[2:end], dts[2:end] ./ dts[2] .* error[2],        label = "O(dt)")
lines!(ax, dts[2:end], (dts[2:end] ./ dts[2]).^2 .* error[2],   label = "O(dt²)")
Legend(fig[1, 2], ax)
fig

## 3. Geometry test

Shorter box (Lx = 20π), propagation direction kz (index 1), isotropic initial E-field perturbation.

In [ ]:
beta_i  = 0.1
mu      = 0.1
epsilon = 1e-7
T       = 0.1

Lx   = 50pi
Nx   = 128
Nv   = 32
vmax = 6 * sqrt(T)

dt = 0.01
Tmax = 50

grid3 = bslLD.Grid([0.0, -vmax, -vmax], [Lx, vmax, vmax], [Nx, Nv, Nv], 1, 1.0, 1)

results3 = [run_simulation(solver, mu, dt, grid3; random_E = true)
            for solver in [bslLD.EMSolverDKNoPol, bslLD.EMSolverDKPol]];

In [ ]:

fig = Figure(size = (900, 400))

for (ind, result) in enumerate(results3)
    k, omega, spec, nk, nw = make_spec(result, dt, grid, Nx)

    ax = Axis(fig[1,ind], xlabel = "k", ylabel = "ω",limits = ((0, 2), (0, 20)))
    heatmap!(ax, k[2:nk], omega[2:nw], spec[2:nw, 2:end]')

    model = [K2Drift,K2DriftPol][ind]

    lines!(ax,[kana(w,1,x->model(x,0,1/10,beta_i)) for w in ws], ws; linestyle = :dash, color = :black)
    lines!(ax,[kana(w,2,x->model(x,0,1/10,beta_i)) for w in ws], ws; linestyle = :dash, color = :black)
end

fig


In [ ]:
fig = Figure()
ax  = Axis(fig[1, 1], xlabel = "t", ylabel = "⟨Ey²⟩", yscale = log10)

labels3 = ["EMSolverDKNoPol", "EMSolverDKPol"]
for (i, result) in enumerate(results3)
    simTime = bslLD.SimulationTime(dt, Tmax)
    Ey2 = map(x -> mean(x .* x), result.Ey)
    lines!(ax, collect(simTime)[1:length(Ey2)], Ey2, label = labels3[i])
end
Legend(fig[1, 2], ax)
fig